# SageMaker Endpoint Monitoring with Amazon Managed Grafana

Creates a Grafana dashboard with panels for GPU utilization, cost attribution, component metrics, and traffic routing.

## Key Patterns for Grafana + CloudWatch

| CloudWatch Native | Grafana Equivalent |
|---|---|
| `SEARCH()` expression | `matchExact: false` on dimension filter |
| `SUM()` metric math | `seriesToColumns` + `calculateField(reduceRow, sum)` transforms |
| `RUNNING_SUM()` | Not supported — use stat panels for totals |
| Series naming | `renameByRegex` transform (label templates don't work with multi-series) |
| Hiding intermediate series | Set `hideFrom` in field defaults, override visible ones back |
| `period=10` for SampleCount=1 | Required — at period=60, SampleCount=6 per GPU (breaks counting) |

## Prerequisites: SageMaker Execution Role Permissions

If running from SageMaker Studio, your execution role needs permissions for IAM, Grafana, and SSO. Run these commands from a **terminal** with IAM admin access (not from this notebook):

```bash
# 1. Create the policy
aws iam create-policy \
  --policy-name SageMakerGrafanaSetup \
  --policy-document '{
    "Version": "2012-10-17",
    "Statement": [
      {
        "Sid": "GrafanaIAMRole",
        "Effect": "Allow",
        "Action": ["iam:CreateRole", "iam:PutRolePolicy", "iam:GetRole", "iam:PassRole"],
        "Resource": "arn:aws:iam::<ACCOUNT-ID>:role/AmazonGrafanaCloudWatchRole"
      },
      {
        "Sid": "GrafanaWorkspace",
        "Effect": "Allow",
        "Action": ["grafana:ListWorkspaces", "grafana:CreateWorkspace", "grafana:DescribeWorkspace", "grafana:UpdatePermissions", "grafana:CreateWorkspaceApiKey"],
        "Resource": "*"
      },
      {
        "Sid": "SSODiscovery",
        "Effect": "Allow",
        "Action": ["sso-admin:ListInstances", "sso:DescribeRegisteredRegions", "sso:GetSharedSsoConfiguration", "sso:ListApplications", "sso:GetManagedApplicationInstance", "sso:CreateManagedApplicationInstance", "identitystore:ListUsers"],
        "Resource": "*"
      }
    ]
  }'

# 2. Attach it to your execution role
aws iam attach-role-policy \
  --role-name <SAGEMAKER-EXECUTION-ROLE-NAME> \
  --policy-arn arn:aws:iam::<ACCOUNT-ID>:policy/SageMakerGrafanaSetup
```

Replace `<ACCOUNT-ID>` and `<SAGEMAKER-EXECUTION-ROLE-NAME>` with your values.

The policy is scoped: IAM actions are limited to the single Grafana role, Grafana/SSO actions are read + create only.

## Step 1: Configuration

In [ ]:
import boto3, json, requests, time

# ========== CONFIGURE THESE ==========
REGION = "us-east-2"
CUSTOM_SUFFIX = "" #enter name
ENDPOINT_NAME = f"enhanced-metrics-{CUSTOM_SUFFIX}"
IC_NAMES = [
    (f"IC-{ENDPOINT_NAME}-gpt-oss", "gpt-oss"),
    (f"IC-{ENDPOINT_NAME}-Qwen", "Qwen"),
]
COST_PER_HOUR_PER_INSTANCE = 5.752  # ml.g5.12xlarge
GPUS_PER_INSTANCE = 4
# ======================================

COST_PER_HOUR_PER_GPU = COST_PER_HOUR_PER_INSTANCE / GPUS_PER_INSTANCE
account_id = boto3.client("sts").get_caller_identity()["Account"]
print(f"Account: {account_id}, Region: {REGION}")
print(f"Cost/hr/GPU: ${COST_PER_HOUR_PER_GPU:.3f}")

## Step 2: Create IAM Role

In [ ]:
iam = boto3.client("iam")
ROLE_NAME = "AmazonGrafanaCloudWatchRole"

trust = {"Version": "2012-10-17", "Statement": [{"Effect": "Allow",
    "Principal": {"Service": "grafana.amazonaws.com"}, "Action": "sts:AssumeRole",
    "Condition": {"StringEquals": {"aws:SourceAccount": account_id}}}]}

try:
    iam.create_role(RoleName=ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust))
    print(f"Created role {ROLE_NAME}")
except iam.exceptions.EntityAlreadyExistsException:
    print(f"Role {ROLE_NAME} exists")

iam.put_role_policy(RoleName=ROLE_NAME, PolicyName="CloudWatchReadAccess",
    PolicyDocument=json.dumps({"Version": "2012-10-17", "Statement": [{"Effect": "Allow",
        "Action": ["cloudwatch:DescribeAlarmsForMetric","cloudwatch:DescribeAlarms",
            "cloudwatch:ListMetrics","cloudwatch:GetMetricData","cloudwatch:GetMetricStatistics",
            "logs:DescribeLogGroups","logs:GetLogGroupFields","logs:StartQuery","logs:StopQuery",
            "logs:GetQueryResults","ec2:DescribeTags","ec2:DescribeInstances","ec2:DescribeRegions",
            "tag:GetResources"], "Resource": "*"}]}))
role_arn = f"arn:aws:iam::{account_id}:role/{ROLE_NAME}"
print(f"Role ARN: {role_arn}")

## Step 3: Create Grafana Workspace (~5 min)

In [ ]:
grafana = boto3.client("grafana", region_name=REGION)
existing = [w for w in grafana.list_workspaces()["workspaces"] if w["name"] == "sagemaker-monitoring"]

if existing:
    WORKSPACE_ID = existing[0]["id"]
    print(f"Existing workspace: {WORKSPACE_ID}")
else:
    resp = grafana.create_workspace(workspaceName="sagemaker-monitoring",
        accountAccessType="CURRENT_ACCOUNT", authenticationProviders=["AWS_SSO"],
        permissionType="SERVICE_MANAGED", workspaceRoleArn=role_arn,
        workspaceDataSources=["CLOUDWATCH"])
    WORKSPACE_ID = resp["workspace"]["id"]
    print(f"Creating {WORKSPACE_ID}...")
    while True:
        s = grafana.describe_workspace(workspaceId=WORKSPACE_ID)["workspace"]["status"]
        print(f"  {s}")
        if s == "ACTIVE": break
        time.sleep(15)

ws = grafana.describe_workspace(workspaceId=WORKSPACE_ID)["workspace"]
GRAFANA_URL = f"https://{ws['endpoint']}"
print(f"URL: {GRAFANA_URL}")

## Step 4: Grant SSO Admin Access

Managed Grafana requires IAM Identity Center (SSO) — there's no plain IAM login option.

**If you don't have SSO set up yet**, enable it first (free, takes ~2 minutes):
1. Go to [IAM Identity Center console](https://console.aws.amazon.com/singlesignon)
2. Click **Enable** (use the default Identity Center directory)
3. Add a user: **Users → Add user** — enter email, first/last name, and set a password
4. Then run this cell

The cell below auto-discovers your SSO instance and grants the first user admin access to Grafana.

In [ ]:
sso_found = False
for r in ["us-east-1", REGION, "us-west-2", "eu-west-1"]:
    try:
        instances = boto3.client("sso-admin", region_name=r).list_instances()["Instances"]
        if instances:
            ids_client = boto3.client("identitystore", region_name=r)
            users = ids_client.list_users(IdentityStoreId=instances[0]["IdentityStoreId"])["Users"]
            print(f"SSO instance found in {r}")
            print(f"Available users:")
            for u in users:
                print(f"  {u['UserName']} — {u['UserId']}")
            user_id = users[0]["UserId"]
            resp = grafana.update_permissions(workspaceId=WORKSPACE_ID,
                updateInstructionBatch=[{"action":"ADD","role":"ADMIN",
                    "users":[{"id":user_id,"type":"SSO_USER"}]}])
            if resp["errors"]:
                print(f"ERROR granting access: {resp['errors']}")
            else:
                print(f"Granted ADMIN to {users[0]['DisplayName']}")
            sso_found = True
            break
    except Exception as e:
        print(f"  Region {r}: {e}")
        continue

if not sso_found:
    print("ERROR: No SSO instance found. Grant access manually via AWS console.")

## Step 5: Add CloudWatch Data Source

In [ ]:
api_key = grafana.create_workspace_api_key(workspaceId=WORKSPACE_ID,
    keyName=f"setup-{int(time.time())}", keyRole="ADMIN", secondsToLive=7200)["key"]
H = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

ds = requests.post(f"{GRAFANA_URL}/api/datasources", headers=H, json={
    "name": "CloudWatch", "type": "cloudwatch", "access": "proxy", "isDefault": True,
    "jsonData": {"authType": "default", "defaultRegion": REGION}}).json()
DS_UID = ds["datasource"]["uid"]
print(f"DataSource UID: {DS_UID}")

## Step 6: Build and Deploy Dashboard

In [ ]:
def cw(ref, ns, metric, dims, stat="Average", period="60", label="", exact=True, hide=False):
    return {"refId": ref, "queryMode": "Metrics", "metricQueryType": 0, "metricEditorMode": 0,
        "region": REGION, "namespace": ns, "metricName": metric,
        "dimensions": {k: ([v] if isinstance(v, str) else v) for k, v in dims.items()},
        "matchExact": exact, "statistic": stat, "period": period, "label": label, "hide": hide}

panels = []
pid = 1

# ===== PANEL 1: Cluster Details =====
# Uses SampleCount at period=10 where each GPU=1, each instance=1
# Sums ALL columns (GPU + instance), subtracts instance count to get Used GPUs
panels.append({
    "id": pid, "type": "timeseries", "title": "Cluster Details — Used GPUs / Free GPUs / Instances",
    "gridPos": {"h": 10, "w": 12, "x": 0, "y": 0},
    "datasource": {"type": "cloudwatch", "uid": DS_UID},
    "targets": [
        cw("A", "/aws/sagemaker/InferenceComponents", "GPUUtilizationNormalized",
           {"EndpointName": ENDPOINT_NAME}, stat="SampleCount", period="10", exact=False),
        cw("B", "/aws/sagemaker/Endpoints", "CPUUtilizationNormalized",
           {"EndpointName": ENDPOINT_NAME, "VariantName": "AllTraffic"},
           stat="SampleCount", period="10", label="Number of Instances"),
    ],
    "transformations": [
        {"id": "seriesToColumns", "options": {"byField": "Time"}},
        {"id": "calculateField", "options": {"mode": "reduceRow", "reduce": {"reducer": "sum"},
            "alias": "All Sum", "replaceFields": False}},
        {"id": "calculateField", "options": {"mode": "binary",
            "binary": {"left": "All Sum", "operator": "-", "right": "Number of Instances"},
            "alias": "Used GPUs", "replaceFields": False}},
        {"id": "calculateField", "options": {"mode": "binary",
            "binary": {"left": "Number of Instances", "operator": "*", "right": str(GPUS_PER_INSTANCE)},
            "alias": "Total GPUs", "replaceFields": False}},
        {"id": "calculateField", "options": {"mode": "binary",
            "binary": {"left": "Total GPUs", "operator": "-", "right": "Used GPUs"},
            "alias": "Free GPUs", "replaceFields": False}},
    ],
    "fieldConfig": {
        "defaults": {"custom": {"drawStyle": "line", "lineInterpolation": "smooth",
            "fillOpacity": 10, "lineWidth": 2,
            "hideFrom": {"legend": True, "tooltip": True, "viz": True}},
            "unit": "short", "decimals": 0},
        "overrides": [
            {"matcher": {"id": "byName", "options": n}, "properties": [
                {"id": "color", "value": {"fixedColor": c, "mode": "fixed"}},
                {"id": "custom.lineWidth", "value": 3},
                {"id": "custom.hideFrom", "value": {"legend": False, "tooltip": False, "viz": False}}
            ]} for n, c in [("Used GPUs", "red"), ("Free GPUs", "green"), ("Number of Instances", "blue")]
        ]
    },
    "options": {"legend": {"displayMode": "list", "placement": "bottom"}, "tooltip": {"mode": "multi"}},
})
pid += 1
print(f"Panel 1: Cluster Details")

### Cost Panels

In [ ]:
# ===== PANELS 2-3: Cost per Hour per IC =====
for ic_name, ic_label in IC_NAMES:
    x = 0 if pid == 2 else 12
    panels.append({
        "id": pid, "type": "timeseries", "title": f"$ Cost/Hour — {ic_label}",
        "gridPos": {"h": 10, "w": 12, "x": x, "y": 10},
        "datasource": {"type": "cloudwatch", "uid": DS_UID},
        "targets": [cw("A", "/aws/sagemaker/InferenceComponents", "GPUUtilizationNormalized",
            {"InferenceComponentName": ic_name}, stat="SampleCount", period="10", exact=False)],
        "transformations": [
            {"id": "seriesToColumns", "options": {"byField": "Time"}},
            {"id": "calculateField", "options": {"mode": "reduceRow", "reduce": {"reducer": "sum"},
                "alias": "GPUs Used", "replaceFields": False}},
            {"id": "calculateField", "options": {"mode": "binary",
                "binary": {"left": "GPUs Used", "operator": "*", "right": str(COST_PER_HOUR_PER_GPU)},
                "alias": "Cost per Hour", "replaceFields": False}},
        ],
        "fieldConfig": {
            "defaults": {"custom": {"drawStyle": "line", "lineInterpolation": "smooth",
                "fillOpacity": 10, "lineWidth": 2}, "unit": "currencyUSD", "decimals": 2},
            "overrides": [
                {"matcher": {"id": "byName", "options": "Cost per Hour"}, "properties": [
                    {"id": "color", "value": {"fixedColor": "orange", "mode": "fixed"}},
                    {"id": "custom.lineWidth", "value": 3}]},
                {"matcher": {"id": "byName", "options": "GPUs Used"}, "properties": [
                    {"id": "custom.hideFrom", "value": {"legend": True, "tooltip": False, "viz": True}}]},
                {"matcher": {"id": "byRegexp", "options": ".*enhanced-metrics.*|GPUUtilization.*"}, "properties": [
                    {"id": "custom.hideFrom", "value": {"legend": True, "tooltip": False, "viz": True}}]},
            ]
        },
        "options": {"legend": {"displayMode": "list", "placement": "bottom"}, "tooltip": {"mode": "multi"}},
    })
    print(f"Panel {pid}: Cost/Hour — {ic_label}")
    pid += 1

### GPU Utilization Panels

In [ ]:
# ===== PANELS 4-7: GPU Compute & Memory per IC =====
y = 20
for ic_name, ic_label in IC_NAMES:
    for col, (metric, title, t1, t2) in enumerate([
        ("GPUUtilizationNormalized", "GPU Compute %", 30, 80),
        ("GPUMemoryUtilizationNormalized", "GPU Memory %", 85, None),
    ]):
        steps = [{"color": "green", "value": None}]
        if t2: steps += [{"color": "orange", "value": t1}, {"color": "red", "value": t2}]
        else: steps.append({"color": "red", "value": t1})

        panels.append({
            "id": pid, "type": "timeseries", "title": f"{title} — {ic_label}",
            "gridPos": {"h": 8, "w": 12, "x": col * 12, "y": y},
            "datasource": {"type": "cloudwatch", "uid": DS_UID},
            "targets": [cw("A", "/aws/sagemaker/InferenceComponents", metric,
                {"InferenceComponentName": ic_name}, stat="Average", period="60", exact=False)],
            "transformations": [{"id": "renameByRegex", "options": {
                "regex": ".*(gpu_\\d+).*", "renamePattern": "$1"}}],
            "fieldConfig": {"defaults": {"custom": {"drawStyle": "line", "lineInterpolation": "smooth",
                "fillOpacity": 10, "lineWidth": 2, "thresholdsStyle": {"mode": "dashed"}},
                "unit": "percent", "min": 0, "max": 100,
                "thresholds": {"mode": "absolute", "steps": steps}}, "overrides": []},
            "options": {"legend": {"displayMode": "list", "placement": "bottom"}, "tooltip": {"mode": "multi"}},
        })
        print(f"Panel {pid}: {title} — {ic_label}")
        pid += 1
    y += 8

### Component Metrics (CPU, Memory, Latency, Errors)

In [ ]:
# ===== PANELS 8-11: Component-level comparison =====
IC_A, IC_B = IC_NAMES[0][0], IC_NAMES[1][0]
LA, LB = IC_NAMES[0][1], IC_NAMES[1][1]

for col, (title, ns, metric, stat, unit) in enumerate([
    ("CPU Utilization", "/aws/sagemaker/InferenceComponents", "CPUUtilizationNormalized", "Average", "percent"),
    ("Memory Utilization", "/aws/sagemaker/InferenceComponents", "MemoryUtilizationNormalized", "Average", "percent"),
    ("Model Latency", "AWS/SageMaker", "ModelLatency", "Average", "µs"),
    ("Total Errors", "AWS/SageMaker", "Invocation5XXErrors", "Sum", "short"),
]):
    panels.append({
        "id": pid, "type": "timeseries", "title": title,
        "gridPos": {"h": 8, "w": 6, "x": col * 6, "y": y},
        "datasource": {"type": "cloudwatch", "uid": DS_UID},
        "targets": [
            cw("A", ns, metric, {"InferenceComponentName": IC_A}, stat=stat, label=LA),
            cw("B", ns, metric, {"InferenceComponentName": IC_B}, stat=stat, label=LB),
        ],
        "fieldConfig": {"defaults": {"custom": {"drawStyle": "line", "lineInterpolation": "smooth",
            "fillOpacity": 10, "lineWidth": 2}, "unit": unit},
            "overrides": [
                {"matcher": {"id": "byName", "options": LA}, "properties": [
                    {"id": "color", "value": {"fixedColor": "blue", "mode": "fixed"}}]},
                {"matcher": {"id": "byName", "options": LB}, "properties": [
                    {"id": "color", "value": {"fixedColor": "orange", "mode": "fixed"}}]},
            ]},
        "options": {"legend": {"displayMode": "list", "placement": "bottom"}, "tooltip": {"mode": "multi"}},
    })
    print(f"Panel {pid}: {title}")
    pid += 1
y += 8

### Traffic Routing Panels

In [ ]:
# ===== PANEL: Total Invocations =====
panels.append({
    "id": pid, "type": "timeseries", "title": f"Total Invocations — {LA} vs {LB}",
    "gridPos": {"h": 8, "w": 24, "x": 0, "y": y},
    "datasource": {"type": "cloudwatch", "uid": DS_UID},
    "targets": [
        cw("A", "AWS/SageMaker", "Invocations", {"InferenceComponentName": IC_A}, stat="Sum", label=LA),
        cw("B", "AWS/SageMaker", "Invocations", {"InferenceComponentName": IC_B}, stat="Sum", label=LB),
    ],
    "fieldConfig": {"defaults": {"custom": {"drawStyle": "line", "lineInterpolation": "smooth",
        "fillOpacity": 15, "lineWidth": 2}, "unit": "short"},
        "overrides": [
            {"matcher": {"id": "byName", "options": LA}, "properties": [
                {"id": "color", "value": {"fixedColor": "blue", "mode": "fixed"}}]},
            {"matcher": {"id": "byName", "options": LB}, "properties": [
                {"id": "color", "value": {"fixedColor": "orange", "mode": "fixed"}}]},
        ]},
    "options": {"legend": {"displayMode": "list", "placement": "bottom"}, "tooltip": {"mode": "multi"}},
})
print(f"Panel {pid}: Total Invocations")
pid += 1
y += 8

# ===== PANELS: Per-Copy Invocations =====
for col, (ic_name, ic_label) in enumerate(IC_NAMES):
    panels.append({
        "id": pid, "type": "timeseries", "title": f"Per-Copy Invocations — {ic_label}",
        "gridPos": {"h": 8, "w": 12, "x": col * 12, "y": y},
        "datasource": {"type": "cloudwatch", "uid": DS_UID},
        "targets": [cw("A", "AWS/SageMaker", "Invocations",
            {"InferenceComponentName": ic_name, "ContainerId": ["*"]},
            stat="Sum", period="60", exact=False)],
        "transformations": [{"id": "renameByRegex", "options": {
            "regex": "^([0-9a-f]{8}).*", "renamePattern": "Copy $1"}}],
        "fieldConfig": {"defaults": {"custom": {"drawStyle": "line", "lineInterpolation": "smooth",
            "fillOpacity": 10, "lineWidth": 2}, "unit": "short"}, "overrides": []},
        "options": {"legend": {"displayMode": "list", "placement": "bottom"}, "tooltip": {"mode": "multi"}},
    })
    print(f"Panel {pid}: Per-Copy — {ic_label}")
    pid += 1

## Step 7: Deploy Dashboard

In [ ]:
resp = requests.post(f"{GRAFANA_URL}/api/dashboards/db", headers=H, json={
    "dashboard": {
        "uid": "sagemaker-ep-monitor",
        "title": "SageMaker Endpoint Monitoring",
        "tags": ["sagemaker", "inference"],
        "timezone": "browser",
        "refresh": "1m",
        "panels": panels,
    },
    "overwrite": True,
})
result = resp.json()
print(f"Status: {result['status']}")
print(f"Dashboard URL: {GRAFANA_URL}{result['url']}")

## Done!

Open the dashboard URL above and log in with your SSO credentials. Set the time range to cover when your endpoint was active.

### Panel Summary

| Panel | What it shows | How it works |
|---|---|---|
| Cluster Details | Used/Free GPUs, instance count | `SampleCount` at 10s period, `reduceRow(sum)` minus instances |
| Cost/Hour | Hourly GPU cost per IC | GPU count × cost_per_hour_per_gpu |
| GPU Compute/Memory | Per-GPU utilization % | `matchExact=false` + `renameByRegex` for clean names |
| CPU/Memory/Latency/Errors | IC comparison | `matchExact=true` with label override, two queries per panel |
| Total Invocations | IC traffic comparison | `matchExact=true`, Sum stat |
| Per-Copy Invocations | Traffic per container | `ContainerId=*` dimension, `renameByRegex` for short names |